In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.session import SparkSession

def get_spark_session(app_name="spark_app_1"):
    """
    Returns an active Spark session if one exists,
    otherwise creates a new Spark session with the given app name.
    """
    try:
        # Try to get the currently active Spark session
        spark = SparkSession.getActiveSession()
        if spark:
            return spark
    except Exception:
        # If no active session is found, ignore the error and create a new one
        pass

    # Create a new Spark session if none is active
    return SparkSession.builder.appName(app_name).getOrCreate()

In [0]:
def read_file(
    file_type: str = 'csv',
    path: str = None,
    header: bool = True,
    inferschema: bool = True,
    sep: str = None,
    linesep: str = None,
    schema: str = None,
    recursivefilelookup: bool = True,
    pathglobalfilter: str = None,
    modifiedbefore: str = None,
    modifiedafter: str = None,
    mode: str = 'PERMISSIVE',
    dateformat: str = 'yyyy-MM-dd',
    timestampformat: str = 'yyyy-MM-dd HH:mm:ss',
    mergeschema: bool = True,
    multiline: bool = False,
    table: str = None,
    is_malformed: bool = False
):
    """
    Reads a file into a Spark DataFrame based on the given file_type.
    Supports CSV, JSON, ORC, Parquet, Delta, and table reads.
    """

    if file_type == 'csv':
        # CSV reader with flexible options
        return spark.read.options(
            header=header,
            inferSchema=inferschema,
            sep=sep,
            lineSep=linesep,
            recursiveFileLookup=recursivefilelookup,
            pathGlobFilter=pathglobalfilter,
            modifiedBefore=modifiedbefore,
            modifiedAfter=modifiedafter,
            mode=mode,
            dateFormat=dateformat,
            timestampFormat=timestampformat
        ).csv(path, schema=schema)

    elif file_type == 'json':
        # JSON reader with multiline and schema options
        return spark.read.options(
            recursiveFileLookup=recursivefilelookup,
            pathGlobFilter=pathglobalfilter,
            multiLine=multiline,
            mode=mode,
            dateFormat=dateformat,
            timestampFormat=timestampformat,
            modifiedBefore=modifiedbefore,
            modifiedAfter=modifiedafter
        ).json(path, schema=schema)

    elif file_type in ['orc', 'parquet', 'delta']:
        # ORC, Parquet, Delta reader with schema merging
        return spark.read.options(
            recursiveFileLookup=recursivefilelookup,
            pathGlobFilter=pathglobalfilter,
            mergeSchema=mergeschema,
            modifiedBefore=modifiedbefore,
            modifiedAfter=modifiedafter
        ).format(file_type).load(path)

    elif file_type == 'table':
        # Read directly from a registered table
        return spark.read.table(table)

    else:
        print('File type not supported')
        return None


In [0]:
%python
##Example Usage:
df_csv = read_file(file_type='csv', path='/mnt/data/sample.csv')

df_json = read_file(file_type='json', path='/mnt/data/sample.json', multiline=True)

df_parquet = read_file(file_type='parquet', path='/mnt/data/sample.parquet')

In [0]:
def write_file(df, file_type: str = 'delta', path: str = None, mode: str = 'overwrite', table: str = None):
    """
    Writes a Spark DataFrame to the specified file type or table.

    Parameters:
    - df: Spark DataFrame to write
    - file_type: 'csv', 'json', 'orc', 'parquet', 'delta', or 'table'
    - path: target path for file-based formats
    - mode: write mode ('overwrite', 'append', 'ignore', 'error')
    - table: target table name if file_type == 'table'
    """

    if file_type in ['csv', 'json', 'orc', 'parquet', 'delta']:
        # Write DataFrame to the given path in the specified format
        return df.write.mode(mode).format(file_type).save(path)

    elif file_type == 'table':
        # Save DataFrame as a managed or external table
        return df.write.mode(mode).saveAsTable(table)

    else:
        print('File type not supported')
        return None

In [0]:
##Example:
#Write to Delta
#write_file(df, file_type='delta', path='/mnt/data/output/delta_table')

#Write to Parquet
#write_file(df, file_type='parquet', path='/mnt/data/output/parquet_table')

#Write to a managed table
#write_file(df, file_type='table', table='my_catalog.my_schema.my_table')

In [0]:
def merge_df(df1, df2, allowmissingcolumns: bool = True):
    """
    Merge two Spark DataFrames by column name.

    Parameters:
    - df1: First Spark DataFrame
    - df2: Second Spark DataFrame
    - allowmissingcolumns: If True, allows union even if one DataFrame
      has columns that the other does not (missing columns will be filled with nulls)

    Returns:
    - A Spark DataFrame resulting from the union of df1 and df2
    """
    return df1.unionByName(df2, allowMissingColumns=allowmissingcolumns)

In [0]:
%python
##Example Usage:
df1 = spark.createDataFrame([(1, "Alice")], ["id", "name"])
df2 = spark.createDataFrame([(2, "Bob", 30)], ["id", "name", "age"])

merged = merge_df(df1, df2)
merged.show()

In [0]:
from pyspark.sql.functions import lit

def add_column_with_default(df, column_name: str, default_value):
    """
    Adds a new column to the DataFrame with a default value.

    Parameters:
    - df: Spark DataFrame
    - column_name: Name of the new column
    - default_value: Value to fill the column (can be str, int, float, bool, None)

    Returns:
    - A new DataFrame with the additional column
    """
    return df.withColumn(column_name, lit(default_value))

In [0]:
#Example Usage
df = spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "name"])

# Add a numeric column
df1 = add_column_with_default(df, "age", 30)

# Add a boolean column
df2 = add_column_with_default(df, "is_active", True)

# Add a string column
df3 = add_column_with_default(df, "role", "user")

df1.show()

In [0]:
def cleansing_func(
    df,
    duplicatedatacolumns: list = None,
    nulldropcolumns: list = [],
    nullstrategy: str = 'any'
):
    """
    Cleanses a Spark DataFrame by:
    1. Removing exact duplicate rows
    2. Dropping duplicates based on specified columns
    3. Dropping rows with nulls based on a strategy

    Parameters:
    - df: Input Spark DataFrame
    - duplicatedatacolumns: List of columns to check for duplicates (default None = all columns)
    - nulldropcolumns: List of columns to check for nulls (default empty list = no null drop)
    - nullstrategy: 'any' (drop if any column is null) or 'all' (drop if all are null)

    Returns:
    - A cleansed Spark DataFrame
    """

    # Step 1: Remove exact duplicate rows
    df1 = df.distinct()

    # Step 2: Drop duplicates based on specific columns (if provided)
    df2 = df1.dropDuplicates(duplicatedatacolumns)

    # Step 3: Drop rows with nulls based on strategy and subset
    df3 = df2.dropna(how=nullstrategy, subset=nulldropcolumns)

    return df3

In [0]:
#Example Usage
df = spark.createDataFrame([
    (1, "Alice", None),
    (2, "Bob", 30),
    (2, "Bob", 30),   # duplicate row
    (3, "Charlie", None)
], ["id", "name", "age"])

cleaned = cleansing_func(
    df,
    duplicatedatacolumns=["id", "name"],
    nulldropcolumns=["age"],
    nullstrategy="any"
)

cleaned.show()

In [0]:
%pip install word2number

from pyspark.sql.functions import udf, col
from pyspark.sql.types import IntegerType
from word2number import w2n

def word_to_num(value):
    """
    Converts a string or numeric input into an integer.
    - If the input is already numeric, returns it as int.
    - If the input is a word (e.g., 'twenty five'), converts it to a number.
    - Returns None if conversion fails.
    """
    try:
        # If already numeric (string of digits or int/float)
        return int(value)
    except Exception:
        try:
            # Convert word representation to number
            return w2n.word_to_num(value.lower())
        except Exception:
            # Return None if conversion fails
            return None

# Register as a Spark UDF
word_to_num_udf = udf(word_to_num, IntegerType())

In [0]:
#Example Usage
df = spark.createDataFrame([("twenty five",), ("100",), ("ten",), ("abc",)], ["input"])

df = df.withColumn("numeric_value", word_to_num_udf(col("input")))
df.show()

In [0]:
def create_temp_view(df, tempviewname: str):
    """
    Creates or replaces a temporary view from a Spark DataFrame.

    Parameters:
    - df: Spark DataFrame
    - tempviewname: Name of the temporary view

    Returns:
    - None (the view is registered in the Spark session)
    """
    return df.createOrReplaceTempView(tempviewname)

In [0]:
#Example Usage
df = spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "name"])

# Register as a temp view
create_temp_view(df, "people_view")

# Query it with SQL
spark.sql("SELECT * FROM people_view WHERE id = 1").show()

In [0]:
def df_from_temp_view(tempviewname: str):
    """
    Returns a Spark DataFrame from a temporary view.

    Parameters:
    - tempviewname: Name of the temporary view created earlier

    Returns:
    - A Spark DataFrame containing all rows and columns from the view
    """
    return spark.sql(f"SELECT * FROM {tempviewname}")

In [0]:
# Assume you already created a temp view
create_temp_view(df, "people_view")

# Retrieve it back as a DataFrame
df_from_view = df_from_temp_view("people_view")
df_from_view.show()